# Qwen3 0.6B Example

Instructions to reproduce:

Base CL: cl/805761842

Inside a g4/hg client:
```
alias colab="/google/bin/releases/grp-ix-team/rapid/colab-cli/cli.par"
colab launch //examples/huggingface_transformers:torch_tpu_colab --use_flaze -a=glp:1x1 --xm_resource_alloc=cml/cml-shared-ml-user
```

Goto (http://colab) and connect to the Xmanager runtime.




In [1]:
import torch

tpu_device = torch.device("tpu")

Successfully renamed PrivateUse1 backend to 'tpu'. Device: tpu
Registered Python module for 'tpu'.


Device type: tpu, Device index: default


In [2]:
# Load model directly
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "/cns/is-d/home/torch-tpu-xm/weights/huggingface/Qwen/Qwen3-0.6B"
)
model_cpu = AutoModelForCausalLM.from_pretrained(
    "/cns/is-d/home/torch-tpu-xm/weights/huggingface/Qwen/Qwen3-0.6B",
    torch_dtype=torch.bfloat16,
)

In [3]:
model_tpu = AutoModelForCausalLM.from_pretrained(
    "/cns/is-d/home/torch-tpu-xm/weights/huggingface/Qwen/Qwen3-0.6B",
    torch_dtype=torch.bfloat16,
)
model_tpu = model_tpu.to(tpu_device)

In [4]:
assert str(model_cpu.device) == "cpu", "model_cpu device should be cpu"
assert model_cpu.dtype == torch.bfloat16, "model_cpu dtype should be bfloat16"
# torch formats this device differently. Verified on mingpt as well.
assert (
    str(model_tpu.device) == f"{tpu_device}:0"
), f"model_tpu device should be {tpu_device}:0, got {model_tpu.device}"
assert model_tpu.dtype == torch.bfloat16, "model_tpu dtype should be bfloat16"

In [7]:
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)
inputs_cpu = inputs.input_ids
inputs_tpu = torch.clone(inputs_cpu).to(tpu_device)

In [10]:
import time


def model_generate(
    model, tokenizer, initial_inputs: torch.Tensor, decode_steps=10
):
  native_device = model.device
  with torch.no_grad():
    start_time = time.time()
    output = model(input_ids=initial_inputs)
    end_time = time.time()
    logits = output.logits

    prefill_time = end_time - start_time
    print(f'Prefill time: {prefill_time * 1000:.2f} ms')
    input_ids = initial_inputs
    start_time = time.time()
    for i in range(decode_steps):
      print(f'Decode step {i}, logits device: {logits.device}')
      # greedy sampling
      logits = logits.to('cpu')
      next_token = (
          torch.argmax(logits[:, -1, :], dim=-1).unsqueeze(-1).to(native_device)
      )
      input_ids = torch.cat([input_ids, next_token], dim=1)
      logits = model(input_ids=input_ids)
      logits = logits.logits
    end_time = time.time()
    decode_time = end_time - start_time
    print(f'Decode time per token: {decode_time * 1000 / decode_steps:.2f} ms')
  # We keep appending all the generated tokens to inputs_ids. The final output
  # is the collected input_ids tensor
  output_ids = input_ids
  output_list = output_ids.tolist()
  output_text = tokenizer.decode(output_list[0], skip_special_tokens=True)

  return output_text

In [12]:
output = model_generate(model_cpu, tokenizer, inputs_cpu, decode_steps=40)
print("-" * 10, "Model Output", "-" * 10)
print(output)

Prefill time: 172.35 ms
Decode step 0, logits device: cpu
Decode step 1, logits device: cpu
Decode step 2, logits device: cpu
Decode step 3, logits device: cpu
Decode step 4, logits device: cpu
Decode step 5, logits device: cpu
Decode step 6, logits device: cpu
Decode step 7, logits device: cpu
Decode step 8, logits device: cpu
Decode step 9, logits device: cpu
Decode step 10, logits device: cpu
Decode step 11, logits device: cpu
Decode step 12, logits device: cpu
Decode step 13, logits device: cpu
Decode step 14, logits device: cpu
Decode step 15, logits device: cpu
Decode step 16, logits device: cpu
Decode step 17, logits device: cpu
Decode step 18, logits device: cpu
Decode step 19, logits device: cpu
Decode step 20, logits device: cpu
Decode step 21, logits device: cpu
Decode step 22, logits device: cpu
Decode step 23, logits device: cpu
Decode step 24, logits device: cpu
Decode step 25, logits device: cpu
Decode step 26, logits device: cpu
Decode step 27, logits device: cpu
Decode

In [14]:
output_tpu = model_generate(model_tpu, tokenizer, inputs_tpu, decode_steps=40)
print("-" * 10, "Model Output", "-" * 10)
print(output_tpu)

Prefill time: 9.64 ms
Decode step 0, logits device: tpu:0
Decode step 1, logits device: tpu:0
Decode step 2, logits device: tpu:0
Decode step 3, logits device: tpu:0
Decode step 4, logits device: tpu:0
Decode step 5, logits device: tpu:0
Decode step 6, logits device: tpu:0
Decode step 7, logits device: tpu:0
Decode step 8, logits device: tpu:0
Decode step 9, logits device: tpu:0
Decode step 10, logits device: tpu:0
Decode step 11, logits device: tpu:0
Decode step 12, logits device: tpu:0
Decode step 13, logits device: tpu:0
Decode step 14, logits device: tpu:0
Decode step 15, logits device: tpu:0
Decode step 16, logits device: tpu:0
Decode step 17, logits device: tpu:0
Decode step 18, logits device: tpu:0
Decode step 19, logits device: tpu:0
Decode step 20, logits device: tpu:0
Decode step 21, logits device: tpu:0
Decode step 22, logits device: tpu:0
Decode step 23, logits device: tpu:0
Decode step 24, logits device: tpu:0
Decode step 25, logits device: tpu:0
Decode step 26, logits dev

In [17]:
output_tpu_2 = model_generate(model_tpu, tokenizer, inputs_tpu, decode_steps=40)
print("-" * 10, "Model Output", "-" * 10)
print(output_tpu_2)

Prefill time: 10.25 ms
Decode step 0, logits device: tpu:0
Decode step 1, logits device: tpu:0
Decode step 2, logits device: tpu:0
Decode step 3, logits device: tpu:0
Decode step 4, logits device: tpu:0
Decode step 5, logits device: tpu:0
Decode step 6, logits device: tpu:0
Decode step 7, logits device: tpu:0
Decode step 8, logits device: tpu:0
Decode step 9, logits device: tpu:0
Decode step 10, logits device: tpu:0
Decode step 11, logits device: tpu:0
Decode step 12, logits device: tpu:0
Decode step 13, logits device: tpu:0
Decode step 14, logits device: tpu:0
Decode step 15, logits device: tpu:0
Decode step 16, logits device: tpu:0
Decode step 17, logits device: tpu:0
Decode step 18, logits device: tpu:0
Decode step 19, logits device: tpu:0
Decode step 20, logits device: tpu:0
Decode step 21, logits device: tpu:0
Decode step 22, logits device: tpu:0
Decode step 23, logits device: tpu:0
Decode step 24, logits device: tpu:0
Decode step 25, logits device: tpu:0
Decode step 26, logits de